In [ ]:
import importlib
import sys
from pathlib import Path

# Add mjosa_code root to path
mjosa_code_root = Path.cwd().parent  # Up to mjosa_code root
sys.path.insert(0, str(mjosa_code_root))

# Import from mjosa_code (NEW structure)
from utils.uhi import georef
from utils.common import config

importlib.reload(georef)
from utils.uhi.georef import *

In [ ]:
# Load 028 transect
transect = load_transect(config.TRANSECT_028_OUTPUT)

transect.list_files()

# Select all 5 files from transect 028
cube = transect.select_files(
    [
        "rad_uhi_20241029_125028_1",
        "rad_uhi_20241029_125028_2",
        "rad_uhi_20241029_125028_3",
        "rad_uhi_20241029_125028_4",
        "rad_uhi_20241029_125028_5",
    ]
)
# cube.describe()

In [ ]:
# Apply illumination correction
cube.apply_illumination_correction_v2()

In [ ]:
cube.apply_spectral_smoothing(method="gaussian", gaussian_sigma=5.0)

In [ ]:
cube.apply_wavelength_filter(wavelength_range=(490, 700))

In [ ]:
cube.apply_spectral_normalization(method="l2")

In [ ]:
def plot_multiple_crops(
    cube,
    crop_regions,
    wavelength_target=660,
    derivative_order=0,
    derivative_window=2,
    crop_width=500,
    crop_aspect_ratio=4,
    display_aspect_ratio=4.0,
    vmin=0.086,
    vmax=0.1,
    use_wavelength_colormap=True,
    flip_axes=True,
    flip_horizontal=True,
    show_file_boundaries=False,
    use_corrected=True,
    **kwargs,
):
    """
    Plot multiple crop regions with the same settings.

    Example:
        crop_regions = [
            {"track": 1258, "slit": 250, "label": "Bomb 2"},
            {"track": 5592, "slit": 700, "label": "Bomb 3"},
        ]

        plot_multiple_crops(
            cube,
            crop_regions,
            wavelength_target=660,
            vmin=0.086,
            vmax=0.1,
        )
    """
    print(f"📊 Plotting {len(crop_regions)} regions...")
    print(f"   Wavelength: {wavelength_target} nm, Derivative: {derivative_order}")
    print(f"   Normalization: vmin={vmin}, vmax={vmax}\n")

    for i, region in enumerate(crop_regions, 1):
        label = region.get("label", f"Region {i}")
        print(
            f"[{i}/{len(crop_regions)}] {label}: track={region['track']}, slit={region['slit']}"
        )

        cube.plot_rgb(
            use_corrected=use_corrected,
            figsize=(30, 8.32),
            flip_axes=flip_axes,
            flip_horizontal=flip_horizontal,
            crop_center_track=region["track"],
            crop_center_slit=region["slit"],
            crop_width=crop_width,
            show_file_boundaries=show_file_boundaries,
            crop_aspect_ratio=crop_aspect_ratio,
            display_aspect_ratio=display_aspect_ratio,
            use_wavelength_colormap=use_wavelength_colormap,
            wavelength_colormap_target=wavelength_target,
            derivative_order=derivative_order,
            derivative_window=derivative_window,
            vmin=vmin,
            vmax=vmax,
            **kwargs,
        )

    print(f"\n✅ All {len(crop_regions)} regions plotted!")

In [ ]:
# 🔧 MONKEYPATCH: Add colorbar orientation support to plot_rgb
# This allows horizontal/vertical colorbar control without kernel restart

import matplotlib.pyplot as plt

# Store the original colorbar function
_original_colorbar = plt.colorbar


def colorbar_with_orientation(*args, **kwargs):
    """
    Wrapper that intercepts colorbar calls and adds orientation support.

    Looks for 'colorbar_orientation' in kwargs and converts it to 'orientation'.
    This allows us to pass colorbar_orientation through **kwargs in plot_multiple_crops!
    """
    # Extract colorbar_orientation if present, default to 'vertical'
    if "colorbar_orientation" in kwargs:
        orientation = kwargs.pop("colorbar_orientation")
        kwargs["orientation"] = orientation
    elif "orientation" not in kwargs:
        kwargs["orientation"] = "vertical"

    return _original_colorbar(*args, **kwargs)


# Apply monkeypatch
plt.colorbar = colorbar_with_orientation

print("✅ Colorbar monkeypatched with orientation support!")
print("   You can now use 'colorbar_orientation' parameter in plot_multiple_crops")
print()
print("💡 Usage examples:")
print("   # Horizontal colorbar:")
print("   plot_multiple_crops(cube, regions, colorbar_orientation='horizontal',")
print("                       colorbar_fraction=0.03, colorbar_pad=0.1)")
print()
print("   # Vertical colorbar (default):")
print("   plot_multiple_crops(cube, regions, colorbar_orientation='vertical',")
print("                       colorbar_fraction=0.006)")

In [ ]:
# 🎨 MONKEYPATCH V2: Non-linear colormap with slow transitions at ends
# Uses a power curve to compress color changes at low/high values
# and expand them in the middle for better contrast

from matplotlib.colors import LinearSegmentedColormap
from utils.ndi_analysis_utils import wavelength_to_rgb
import utils.ndi_analysis_utils as ndi_utils
import numpy as np


def create_wavelength_colormap_nonlinear(wavelength, darkness_factor=0.3):
    """
    NON-LINEAR VERSION: Create a colormap with perceptually better distribution

    Uses a power curve (x^2.5) to create:
    - Slow transition at low values (stays white longer)
    - Fast transition in middle (good contrast)
    - Slow transition at high values (gradual to dark)

    This gives better visual accuracy for identifying features!
    """
    # Get the RGB color for this wavelength
    wl_rgb = wavelength_to_rgb(wavelength)

    # Create color stops
    white = (1.0, 1.0, 1.0)
    light_color = tuple((c + 2 * 1.0) / 3 for c in wl_rgb)  # Mix 1/3 color + 2/3 white
    bright_color = wl_rgb
    dark_color = tuple(c * darkness_factor for c in wl_rgb)

    # Create many positions using a power curve for non-linear mapping
    # x^2.5 compresses the ends and expands the middle
    n_points = 100
    linear_positions = np.linspace(0, 1, n_points)

    # Apply power curve: x^2.5 gives good balance
    # Lower exponent (2.0) = more linear
    # Higher exponent (3.0) = more contrast in middle
    power = 2.5
    nonlinear_positions = linear_positions**power

    # Interpolate colors based on non-linear positions
    colors_interp = []
    for pos in nonlinear_positions:
        if pos < 0.4:
            # White to light color (40% of range)
            alpha = pos / 0.4
            color = tuple(
                w * (1 - alpha) + l * alpha for w, l in zip(white, light_color)
            )
        elif pos < 0.7:
            # Light to bright (30% of range)
            alpha = (pos - 0.4) / 0.3
            color = tuple(
                l * (1 - alpha) + b * alpha for l, b in zip(light_color, bright_color)
            )
        else:
            # Bright to dark (30% of range)
            alpha = (pos - 0.7) / 0.3
            color = tuple(
                b * (1 - alpha) + d * alpha for b, d in zip(bright_color, dark_color)
            )
        colors_interp.append(color)

    # Create colormap with all interpolated colors
    cmap = LinearSegmentedColormap.from_list(
        f"wl_{wavelength}nm_nonlinear",
        list(zip(linear_positions, colors_interp)),
        N=256,
    )

    return cmap


# Replace the function with non-linear version
ndi_utils.create_wavelength_colormap = create_wavelength_colormap_nonlinear

print("✅ Wavelength colormap function patched with NON-LINEAR scaling!")
print("   Power curve (x^2.5) applied for better perceptual accuracy")
print("   - Slow transitions at low values (more white, less noise)")
print("   - Fast transitions in middle (good feature contrast)")
print("   - Slow transitions at high values (gradual saturation)")
print("\n💡 Adjust 'power' variable in cell to tweak (2.0=linear, 3.0=more contrast)")

In [ ]:
# 🧪 TEST: Re-plot with NON-LINEAR colormap
# This is a copy of cell 8 for quick testing with the power curve

# Define your crop regions once
crop_regions = [
    {"track": 5160, "slit": 613, "label": "Bomb 1"},
    {"track": 1258, "slit": 250, "label": "Bomb 2"},
    {"track": 5592, "slit": 700, "label": "Bomb 3"},
    {"track": 717, "slit": 385, "label": "Dark pits"},
    {"track": 4026, "slit": 582, "label": "Between 1-2"},
]

# Plot all regions with NON-LINEAR colormap (power curve = 2.5)
plot_multiple_crops(
    cube,
    crop_regions,
    crop_width=500,
    derivative_order=0,
    vmin=0.086,
    vmax=0.108,
    wavelength_target=677,
    colorbar_orientation="horizontal",  # ⬅️ HORIZONTAL!
    colorbar_fraction=0.05,  # Height (taller for horizontal)
    colorbar_pad=0.1,  # Space below plot
)

plot_multiple_crops(
    cube,
    crop_regions,
    crop_width=500,
    derivative_order=1,
    vmin=-0.000365,
    vmax=0.000427,
    wavelength_target=660,
    colorbar_orientation="horizontal",  # ⬅️ HORIZONTAL!
    colorbar_fraction=0.05,  # Height (taller for horizontal)
    colorbar_pad=0.1,  # Space below plot
)

plot_multiple_crops(
    cube,
    crop_regions,
    crop_width=500,
    derivative_order=0,
    vmin=0.0793,
    vmax=0.105,
    wavelength_target=490,
    colorbar_orientation="horizontal",  # ⬅️ HORIZONTAL!
    colorbar_fraction=0.05,  # Height (taller for horizontal)
    colorbar_pad=0.1,  # Space below plot
)

plot_multiple_crops(
    cube,
    crop_regions,
    crop_width=500,
    derivative_order=0,
    vmin=0.0855,
    vmax=0.095,
    wavelength_target=610,
    colorbar_orientation="horizontal",  # ⬅️ HORIZONTAL!
    colorbar_fraction=0.05,  # Height (taller for horizontal)
    colorbar_pad=0.1,  # Space below plot
)

In [ ]:
# 🧪 TEST: Colorbar Orientation (Horizontal vs Vertical)
# Make sure to run the colorbar monkeypatch cell (cell 8) first!

# Example 1: HORIZONTAL colorbar at bottom of plot
plot_multiple_crops(
    cube,
    [{"track": 5160, "slit": 613, "label": "Bomb 1"}],
    crop_width=500,
    derivative_order=0,
    vmin=0.086,
    vmax=0.108,
    wavelength_target=677,
    # NEW COLORBAR CONTROLS:
    colorbar_orientation="horizontal",  # ⬅️ HORIZONTAL!
    colorbar_fraction=0.05,  # Height (taller for horizontal)
    colorbar_pad=0.1,  # Space below plot
    # colorbar_shrink=0.8,  # 80% of plot width
)

In [ ]:
# 🧪 More examples with different orientations

# Example 2: Thick VERTICAL colorbar (right side)
# plot_multiple_crops(
#     cube,
#     [{"track": 1258, "slit": 250, "label": "Bomb 2"}],
#     crop_width=500,
#     derivative_order=0,
#     vmin=0.086,
#     vmax=0.108,
#     wavelength_target=677,
#     colorbar_orientation="vertical",  # ⬅️ VERTICAL (default)
#     colorbar_fraction=0.02,  # Width (thicker)
#     colorbar_shrink=1.0,  # Full height
# )

# Example 3: Compare derivative with horizontal colorbar
# plot_multiple_crops(
#     cube,
#     crop_regions,
#     crop_width=500,
#     derivative_order=1,  # First derivative
#     vmin=-0.000365,
#     vmax=0.000427,
#     wavelength_target=660,
#     colorbar_orientation="horizontal",
#     colorbar_fraction=0.025,
#     colorbar_pad=0.08,
# )

---

---

---

---

---

---

---

---

# greyscale for 057


In [ ]:
stop

In [ ]:
import importlib
import sys
from pathlib import Path

# Add mjosa_code root to path
mjosa_code_root = Path.cwd().parent  # Up to mjosa_code root
sys.path.insert(0, str(mjosa_code_root))

# Import from mjosa_code (NEW structure)
from utils.uhi import georef
from utils.common import config

importlib.reload(georef)
from utils.uhi.georef import *

In [ ]:
# ========== TRANSECT 057 (DEFAULT) ==========
# Load transect 057
transect = load_transect(config.TRANSECT_057_OUTPUT)
transect.list_files()

# Select file from transect 057
cube = transect.select_files(["rad_uhi_20241029_115057_5"])
cube.describe()

In [ ]:
cube.apply_illumination_correction_v2()

In [ ]:
cube.apply_spectral_smoothing(method="gaussian", gaussian_sigma=5.0)

In [ ]:
cube.apply_wavelength_filter(wavelength_range=(490, 700))

In [ ]:
cube.apply_spectral_normalization(method="l2")

In [ ]:
cube.plot_georef(
    use_corrected=True,
    track_start=config.UHI_TRACK_RANGE[0],
    track_end=config.UHI_TRACK_RANGE[1],
    use_wavelength_colormap=True,
    wavelength_colormap_target=677,
    # vmin=0.09,
    # vmax=0.15,
    # make it use the same minmax as the other greyscales
    vmin=0.086,
    vmax=0.1,
    # Let it auto-compute to see actual data range
    colorbar_fraction=0.006,  # Width
    colorbar_pad=0.04,  # Padding (now works!)
    colorbar_shrink=0.5,  # Height (now works!)
    figsize=(50, 20),
)